# 01 — Explore the Kaggriculture environment

Level 0 foundations step (see `PROGRESS.md`): play random/starter episodes,
inspect a raw `obs` dict, and inspect a `replay.json`. No modeling here —
purely characterizing the environment API before writing `state.py` /
`actions.py` / `evaluate.py`.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from kaggle_environments import make

from kaggriculture_bot.config import ANIMALS, CROPS, MARKET_PARAMS, PRODUCTS

print("Crops:", list(CROPS.keys()))
print("Animals:", list(ANIMALS.keys()))
print("Products:", list(PRODUCTS))

Crops: ['WHEAT', 'CARROT', 'TOMATO', 'STRAWBERRY', 'MELON']
Animals: ['GOOSE', 'COW', 'SHEEP']
Products: ['WHEAT', 'CARROT', 'TOMATO', 'STRAWBERRY', 'MELON', 'EGG', 'MILK', 'WOOL', 'FERTILIZER']


## Run a full-length random-vs-random episode

Default config: `episodeSteps=720` (24 turns/day x 30 days).

In [2]:
env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
env.run(["random", "random"])

print("Recorded steps:", len(env.steps))
final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s['reward']}, status={s['status']}")

Recorded steps: 720
Player 0: reward=0.0, status=DONE
Player 1: reward=0.0, status=DONE


## Inspect a raw `obs` dict at step 0

Top-level keys and the per-player-visible substructures, per the
Observation Format documented in `README.md` / `AGENTS.md`.

In [3]:
obs0 = env.steps[0][0]["observation"]
print("Top-level obs keys:", list(obs0.keys()))
print()
print("farms[0]:")
print(json.dumps(obs0["farms"][0], indent=2, default=str))

Top-level obs keys: ['remainingOverageTime', 'step', 'player', 'farms', 'private', 'market', 'town', 'day', 'hour']

farms[0]:
{
  "money": 3000.0,
  "tiles": [
    [
      null,
      null,
      null,
      null,
      null,
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED"
    ],
    [
      null,
      null,
      null,
      null,
      null,
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED"
    ],
    [
      null,
      null,
      null,
      null,
      null,
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED"
    ],
    [
      null,
      null,
      null,
      null,
      null,
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED"
    ],
    [
      null,
      null,
      null,
      null,
      null,
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED"
    ],
    [
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LOCKED",
      "LO

In [4]:
print("market:")
print(json.dumps(obs0["market"], indent=2, default=str))
print()
print("private:")
print(json.dumps(obs0["private"], indent=2, default=str))
print()
print("town:", obs0["town"])

market:
{
  "inventory": {
    "WHEAT": 10000,
    "CARROT": 10000,
    "TOMATO": 10000,
    "STRAWBERRY": 10000,
    "MELON": 10000,
    "EGG": 10000,
    "MILK": 10000,
    "WOOL": 10000,
    "FERTILIZER": 10000
  },
  "prices": {
    "WHEAT": 25,
    "CARROT": 35,
    "TOMATO": 60,
    "STRAWBERRY": 120,
    "MELON": 250,
    "EGG": 50,
    "MILK": 160,
    "WOOL": 200,
    "FERTILIZER": 100
  }
}

private:
{
  "shed": {
    "WHEAT": 0,
    "CARROT": 0,
    "TOMATO": 0,
    "STRAWBERRY": 0,
    "MELON": 0,
    "EGG": 0,
    "MILK": 0,
    "WOOL": 0,
    "FERTILIZER": 0,
    "GOOSE": 0,
    "COW": 0,
    "SHEEP": 0
  },
  "seeds": {
    "WHEAT": 0,
    "CARROT": 0,
    "TOMATO": 0,
    "STRAWBERRY": 0,
    "MELON": 0
  },
  "inventories": [
    {}
  ]
}

town: {'unlocked_shops': []}


## Find example tile dicts of each kind mid-game

Scan every step's `tiles` grid for player 0 until we've seen one example of
each documented tile kind: `None`, `"LOCKED"`, a plant, a weed, and an
animal structure (with and without an animal placed).

In [5]:
def tile_kind(tile):
    if tile is None:
        return "EMPTY"
    if tile == "LOCKED":
        return "LOCKED"
    if tile["kind"] == "PLANT":
        return f"PLANT:{tile['crop']}"
    if tile["kind"] == "WEED":
        return "WEED"
    if tile["kind"] in ("COOP", "PASTURE"):
        return f"{tile['kind']}:{tile['animal'] or 'empty'}"
    return f"UNKNOWN:{tile}"


examples = {}
for step in env.steps:
    obs = step[0]["observation"]
    for row in obs["farms"][0]["tiles"]:
        for tile in row:
            kind = tile_kind(tile)
            if kind not in examples:
                examples[kind] = tile

for kind, tile in examples.items():
    print(kind, "->", tile)

EMPTY -> None
LOCKED -> LOCKED
PLANT:STRAWBERRY -> {'kind': 'PLANT', 'crop': 'STRAWBERRY', 'planted_day': 2, 'watered_today': False, 'consecutive_unwatered': 1, 'yield_units': 0, 'max_lifespan_step': -1, 'fertilized_until_day': -1}
PLANT:WHEAT -> {'kind': 'PLANT', 'crop': 'WHEAT', 'planted_day': 2, 'watered_today': False, 'consecutive_unwatered': 1, 'yield_units': 1, 'max_lifespan_step': 168, 'fertilized_until_day': -1}
WEED -> {'kind': 'WEED'}
PLANT:TOMATO -> {'kind': 'PLANT', 'crop': 'TOMATO', 'planted_day': 4, 'watered_today': False, 'consecutive_unwatered': 1, 'yield_units': 0, 'max_lifespan_step': -1, 'fertilized_until_day': -1}
PLANT:CARROT -> {'kind': 'PLANT', 'crop': 'CARROT', 'planted_day': 6, 'watered_today': False, 'consecutive_unwatered': 1, 'yield_units': 1, 'max_lifespan_step': 240, 'fertilized_until_day': -1}
PLANT:MELON -> {'kind': 'PLANT', 'crop': 'MELON', 'planted_day': 11, 'watered_today': False, 'consecutive_unwatered': 1, 'yield_units': 1, 'max_lifespan_step': 576,

## Run random vs. starter, compare

Sanity check that the built-in `"starter"` baseline beats `"random"`, which
is the first-order signal that the eval harness (`evaluate.py`, Level 0)
will need to reproduce.

In [6]:
env2 = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
env2.run(["random", "starter"])

final2 = env2.steps[-1]
names = ["random", "starter"]
for name, s in zip(names, final2):
    print(f"{name}: reward={s['reward']}, status={s['status']}")

random: reward=0.0, status=DONE
starter: reward=3503.0, status=DONE


## Dump and reload a replay.json

Confirms the round-trip used for `kaggle competitions replay` analysis
(`AGENTS.md` "Download Replays and Logs").

In [7]:
replay_path = Path.cwd().parent / "replays" / "01_explore_random_vs_starter.json"
with open(replay_path, "w") as f:
    json.dump(env2.toJSON(), f)

with open(replay_path) as f:
    replay = json.load(f)

print("Saved to:", replay_path)
print("Top-level replay keys:", list(replay.keys()))
print("Number of steps in replay:", len(replay["steps"]))

Saved to: C:\Users\alher\Desktop\kaggriculture\replays\01_explore_random_vs_starter.json
Top-level replay keys: ['id', 'name', 'title', 'description', 'version', 'module_version', 'configuration', 'specification', 'steps', 'rewards', 'statuses', 'schema_version', 'info']
Number of steps in replay: 720


## Findings

Summarized in `PROGRESS.md` once this notebook has been run end to end.